# 🎙️ DeepBrief.AI — Stage 2: Speech-to-Text Transcription

Transcribes each saved `.wav` sample using **Groq API (whisper-large-v3)** and evaluates
accuracy against AMI ground truth using **normalised WER**.

Input: `data/samples_manifest.json` + `.wav` files from Stage 1
Output: `data/transcripts.json` — one transcript per sample with WER scores

**Run Stage 1 first. Make sure `data/samples_manifest.json` exists.**

## ⚙️ Cell 1 — Imports & Configuration

In [1]:
import os
import re
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# ─── Paths ────────────────────────────────────────────────────────────────────
SAMPLES_MANIFEST = Path("data/samples_manifest.json")
TRANSCRIPTS_PATH = Path("data/transcripts.json")

# ─── Groq config ──────────────────────────────────────────────────────────────
GROQ_MODEL       = "whisper-large-v3"
MAX_FILE_SIZE_MB = 25       # Groq free tier hard limit
RATE_LIMIT_DELAY = 2        # seconds between requests (free tier: 20 req/min)

groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    raise EnvironmentError("GROQ_API_KEY not found — check your .env file")

client = Groq(api_key=groq_key)
print(f"✓ Groq client ready  |  model: {GROQ_MODEL}")
print(f"✓ API key: {groq_key[:8]}...")

✓ Groq client ready  |  model: whisper-large-v3
✓ API key: gsk_lwX8...


## 🧹 Cell 2 — Text Normalisation for WER

Raw WER on the AMI ground truth produces artificially inflated scores for three reasons:

1. **Capitalisation** — AMI ground truth is `ALL CAPS`; Whisper returns normal cased text.
   Lowercasing both fixes this completely.

2. **Disfluency repetitions** — The AMI transcripts faithfully capture how people actually
   speak, including false starts like `IT'LL IT'LL` or `IF YOU IF YOU`. Whisper intelligently
   collapses these into one clean word as a fluent speaker would. Penalising Whisper for this
   is unfair — the model is more readable, not less accurate.

3. **Spelled-out words** — AMI writes `S. S. H.` where Whisper transcribes `SSH`. Same word,
   different notation. The normaliser strips punctuation and collapses multi-char sequences.

The `normalise()` function below applies all three fixes to both the hypothesis (Whisper output)
and the reference (AMI ground truth) before WER is computed. This is standard practice in
ASR evaluation and is the approach used in published Whisper benchmark papers.

In [3]:
import re

def normalise(text: str) -> str:
    """Normalise text for fair WER comparison between Whisper output and AMI ground truth."""

    # 1. Lowercase
    text = text.lower()

    # 2. Strip punctuation (commas, periods, apostrophes etc.)
    text = re.sub(r"[^\w\s]", "", text)

    # 3. Collapse consecutive duplicate words  (e.g. "it'll it'll" → "it'll")
    #    Matches any word immediately followed by itself (with optional whitespace between)
    text = re.sub(r"\b(\w+)(?:\s+\1)+\b", r"\1", text)

    # 4. Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
# ── Sanity check ──────────────────────────────────────────────────────────────
tests = [
    ("YEAH IT'LL IT'LL PLAY THEM IN SOME ORDER",
     "Yeah, it'll play them in some order"),
    ("IT'S YEAH I MEAN THE WAVE DATA ARE OBVIOUSLY NOT GONNA GET OFF THERE",
     "because yeah, I mean, the wave data are obviously not going to get off there completely."),
]

print("Normalisation sanity checks:")
print("-" * 60)
for ref, hyp in tests:
    print(f"  REF raw : {ref}")
    print(f"  HYP raw : {hyp}")
    print(f"  REF norm: {normalise(ref)}")
    print(f"  HYP norm: {normalise(hyp)}")
    print()

Normalisation sanity checks:
------------------------------------------------------------
  REF raw : YEAH IT'LL IT'LL PLAY THEM IN SOME ORDER
  HYP raw : Yeah, it'll play them in some order
  REF norm: yeah itll play them in some order
  HYP norm: yeah itll play them in some order

  REF raw : IT'S YEAH I MEAN THE WAVE DATA ARE OBVIOUSLY NOT GONNA GET OFF THERE
  HYP raw : because yeah, I mean, the wave data are obviously not going to get off there completely.
  REF norm: its yeah i mean the wave data are obviously not gonna get off there
  HYP norm: because yeah i mean the wave data are obviously not going to get off there completely



## 📂 Cell 3 — Load Manifest from Stage 1

In [4]:
if not SAMPLES_MANIFEST.exists():
    raise FileNotFoundError("data/samples_manifest.json not found — run Stage 1 first.")

with open(SAMPLES_MANIFEST) as f:
    manifest = json.load(f)

print(f"✓ Loaded {len(manifest)} samples from manifest\n")

for s in manifest:
    size_mb = Path(s["audio_path"]).stat().st_size / 1e6
    flag = "⚠ TOO LARGE" if size_mb > MAX_FILE_SIZE_MB else "✓"
    print(f"  {flag}  {s['sample_id']}  |  {s['duration_seconds']}s  |  {size_mb:.2f} MB  |  {s['meeting_id']}")

✓ Loaded 20 samples from manifest

  ✓  EN2001a_clip00  |  4.21s  |  0.13 MB  |  EN2001a
  ✓  EN2001a_clip01  |  1.63s  |  0.05 MB  |  EN2001a
  ✓  EN2001b_clip00  |  4.21s  |  0.13 MB  |  EN2001b
  ✓  EN2001b_clip01  |  1.77s  |  0.06 MB  |  EN2001b
  ✓  EN2001d_clip00  |  0.95s  |  0.03 MB  |  EN2001d
  ✓  EN2001d_clip01  |  2.74s  |  0.09 MB  |  EN2001d
  ✓  EN2001e_clip00  |  0.88s  |  0.03 MB  |  EN2001e
  ✓  EN2001e_clip01  |  3.4s  |  0.11 MB  |  EN2001e
  ✓  EN2003a_clip00  |  4.26s  |  0.14 MB  |  EN2003a
  ✓  EN2003a_clip01  |  0.29s  |  0.01 MB  |  EN2003a
  ✓  EN2004a_clip00  |  3.38s  |  0.11 MB  |  EN2004a
  ✓  EN2004a_clip01  |  2.09s  |  0.07 MB  |  EN2004a
  ✓  EN2005a_clip00  |  2.24s  |  0.07 MB  |  EN2005a
  ✓  EN2005a_clip01  |  0.84s  |  0.03 MB  |  EN2005a
  ✓  EN2006a_clip00  |  4.49s  |  0.14 MB  |  EN2006a
  ✓  EN2006a_clip01  |  0.24s  |  0.01 MB  |  EN2006a
  ✓  EN2006b_clip00  |  4.76s  |  0.15 MB  |  EN2006b
  ✓  EN2006b_clip01  |  1.12s  |  0.04 MB  |  EN

## 🗣️ Cell 4 — Transcribe with Groq Whisper

Sends each `.wav` to `whisper-large-v3` via the Groq API.
- Files over 25 MB are skipped (Groq free tier limit)
- 2s delay between requests stays within the 20 req/min free tier rate limit
- Results saved incrementally so a mid-run failure doesn't lose completed work

In [7]:
transcripts = []

print(f"Transcribing {len(manifest)} samples via Groq ({GROQ_MODEL})...\n")
print("=" * 60)

for i, sample in enumerate(manifest):
    audio_path = Path(sample["audio_path"])
    sample_id  = sample["sample_id"]

    size_mb = audio_path.stat().st_size / 1e6
    if size_mb > MAX_FILE_SIZE_MB:
        print(f"  [{i+1}/{len(manifest)}] {sample_id} — SKIPPED ({size_mb:.1f} MB > 25 MB limit)")
        continue

    print(f"  [{i+1}/{len(manifest)}] {sample_id} — {sample['duration_seconds']}s — sending to Groq...")

    try:
        with open(audio_path, "rb") as audio_file:
            response = client.audio.transcriptions.create(
                file=(audio_path.name, audio_file, "audio/wav"),
                model=GROQ_MODEL,
                response_format="verbose_json",
                language="en",
            )

        transcript_text = response.text.strip()

        entry = {
            "sample_id":         sample_id,
            "audio_path":        str(audio_path),
            "duration_seconds":  sample["duration_seconds"],
            "speaker_id":        sample["speaker_id"],
            "meeting_id":        sample["meeting_id"],
            "begin_time":        sample["begin_time"],
            "end_time":          sample["end_time"],
            "ground_truth_text": sample["ground_truth_text"],
            "transcript":        transcript_text,
            "model":             GROQ_MODEL,
        }
        transcripts.append(entry)
        print(f'            ✓ Transcript: "{transcript_text[:80]}..."')

    except Exception as e:
        print(f"            ✗ Error: {e}")
        transcripts.append({**sample, "transcript": None, "error": str(e), "model": GROQ_MODEL})

    if i < len(manifest) - 1:
        time.sleep(RATE_LIMIT_DELAY)

Transcribing 20 samples via Groq (whisper-large-v3)...

  [1/20] EN2001a_clip00 — 4.21s — sending to Groq...
            ✓ Transcript: "If you SSH in there, there's this big warning about doing nothing at all in the ..."
  [2/20] EN2001a_clip01 — 1.63s — sending to Groq...
            ✓ Transcript: "Hardly any...."
  [3/20] EN2001b_clip00 — 4.21s — sending to Groq...
            ✓ Transcript: "This is how much time you spend just getting the right software going...."
  [4/20] EN2001b_clip01 — 1.77s — sending to Groq...
            ✓ Transcript: "Yeah, yeah, probably...."
  [5/20] EN2001d_clip00 — 0.95s — sending to Groq...
            ✓ Transcript: "No, no...."
  [6/20] EN2001d_clip01 — 2.74s — sending to Groq...
            ✓ Transcript: "It's ready to get started...."
  [7/20] EN2001e_clip00 — 0.88s — sending to Groq...
            ✓ Transcript: "Yeah, okay, cool...."
  [8/20] EN2001e_clip01 — 3.4s — sending to Groq...
            ✓ Transcript: "but it's probably not a simple thing..

In [8]:
# Save all transcripts to disk
with open(TRANSCRIPTS_PATH, "w") as f:
    json.dump(transcripts, f, indent=2)

print("=" * 60)
print(f"✓ Transcripts saved → {TRANSCRIPTS_PATH}")

✓ Transcripts saved → data\transcripts.json


## 📋 Cell 6 — Results Summary

Side-by-side comparison of Whisper output vs AMI ground truth.

In [9]:
print("=" * 60)
print("Transcription Results Summary")
print("=" * 60)

success = [t for t in transcripts if t.get("transcript")]
failed  = [t for t in transcripts if not t.get("transcript")]

print(f"  Successful : {len(success)}/{len(transcripts)}")
print(f"  Failed     : {len(failed)}/{len(transcripts)}")
print()

for t in transcripts:
    print(f"  ── {t['sample_id']}  |  speaker: {t['speaker_id']}  |  {t['duration_seconds']}s  |  meeting: {t['meeting_id']}")
    if t.get("transcript"):
        print(f"     Whisper : {t['transcript'][:100]}")
        print(f"     Ground  : {t['ground_truth_text'][:100]}")
    else:
        print(f"     ERROR   : {t.get('error')}")
    print()

Transcription Results Summary
  Successful : 20/20
  Failed     : 0/20

  ── EN2001a_clip00  |  speaker: MEO069  |  4.21s  |  meeting: EN2001a
     Whisper : If you SSH in there, there's this big warning about doing nothing at all in the gateway machine.
     Ground  : IF YOU IF YOU S. S. H. AND THEY HAVE THIS BIG WARNING ABOUT DOING NOTHING AT ALL IN THE GATEWAY MACH

  ── EN2001a_clip01  |  speaker: MEE068  |  1.63s  |  meeting: EN2001a
     Whisper : Hardly any.
     Ground  : I'VE GOTTEN MM HARDLY ANY

  ── EN2001b_clip00  |  speaker: MEO069  |  4.21s  |  meeting: EN2001b
     Whisper : This is how much time you spend just getting the right software going.
     Ground  : OH THIS IS HOW MUCH TIME YOU SPEND JUST GETTING THE RIGHT SOFTWARE GOING

  ── EN2001b_clip01  |  speaker: FEO065  |  1.77s  |  meeting: EN2001b
     Whisper : Yeah, yeah, probably.
     Ground  : YEAH YEAH PROBABLY

  ── EN2001d_clip00  |  speaker: MEO069  |  0.95s  |  meeting: EN2001d
     Whisper : No, no.
     

## 📐 Cell 7 — Normalised Word Error Rate (WER)

Computes WER using the `normalise()` function from Cell 2 applied to **both** the Whisper
output and the AMI ground truth before scoring. This removes the artificial penalty from
capitalisation differences, disfluency repetitions, and spelling notation mismatches.

Both raw WER (no normalisation) and normalised WER are reported so the improvement
from normalisation is visible and can be discussed in the project report.

Target from project spec: **WER < 25%**

In [10]:
from jiwer import wer

evaluable = [
    t for t in transcripts
    if t.get("transcript") and t.get("ground_truth_text")
]

if not evaluable:
    print("No evaluable samples — need both transcript and ground truth.")
else:
    print("=" * 60)
    print("Word Error Rate — Raw vs Normalised")
    print("Target: < 25%")
    print("=" * 60)
    print(f"  {'Sample':<30} {'Raw WER':>10} {'Norm WER':>10}  Status")
    print(f"  {'-'*30} {'-'*10} {'-'*10}  ------")

    raw_scores  = []
    norm_scores = []

    for t in evaluable:
        ref_raw  = t["ground_truth_text"]
        hyp_raw  = t["transcript"]

        raw_score  = wer(ref_raw.lower(),         hyp_raw.lower())
        norm_score = wer(normalise(ref_raw),       normalise(hyp_raw))

        raw_scores.append(raw_score)
        norm_scores.append(norm_score)

        status = "✓" if norm_score < 0.25 else "✗"
        print(f"  {t['sample_id']:<30} {raw_score:>9.1%}  {norm_score:>9.1%}  {status}")

    avg_raw  = sum(raw_scores)  / len(raw_scores)
    avg_norm = sum(norm_scores) / len(norm_scores)
    status   = "✓ PASS" if avg_norm < 0.25 else "✗ FAIL"

    print(f"  {'─'*30} {'─'*10} {'─'*10}")
    print(f"  {'AVERAGE':<30} {avg_raw:>9.1%}  {avg_norm:>9.1%}  {status} (target < 25%)")
    print()
    print(f"  Normalisation reduced average WER by {avg_raw - avg_norm:.1%}")
    print()
    print("✅ Stage 2 complete. Proceed to stage3.ipynb")

Word Error Rate — Raw vs Normalised
Target: < 25%
  Sample                            Raw WER   Norm WER  Status
  ------------------------------ ---------- ----------  ------
  EN2001a_clip00                     40.9%      33.3%  ✗
  EN2001a_clip01                     80.0%      60.0%  ✗
  EN2001b_clip00                     14.3%       7.1%  ✓
  EN2001b_clip01                    100.0%       0.0%  ✓
  EN2001d_clip00                    100.0%      50.0%  ✗
  EN2001d_clip01                     20.0%       0.0%  ✓
  EN2001e_clip00                    100.0%       0.0%  ✓
  EN2001e_clip01                     40.0%      30.0%  ✗
  EN2003a_clip00                     27.3%       9.1%  ✓
  EN2003a_clip01                    100.0%       0.0%  ✓
  EN2004a_clip00                     15.4%       0.0%  ✓
  EN2004a_clip01                    100.0%       0.0%  ✓
  EN2005a_clip00                     66.7%      50.0%  ✗
  EN2005a_clip01                     33.3%       0.0%  ✓
  EN2006a_clip00          